# 01 Bronze manifest
I keep every downloaded file untouched and record a SHA-256 for each, so that anyone can later prove which bytes this analysis used. Nothing is transformed here.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import common as c
pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.width = 200
from datetime import datetime, timezone

In [2]:
# The URLs I downloaded from. The monthly sitrep URLs are in urls.txt, written when I downloaded them.
urls = {
    'AE2526_CQI_Open_Data_AR.csv': 'https://files.digital.nhs.uk/96/785BF6/AE2526_CQI_Open_Data_AR.csv',
    'AE2526_ECDS_MSitAE_Tables.xlsx': 'https://files.digital.nhs.uk/08/D5507A/AE2526_ECDS_MSitAE_Tables.xlsx',
    'AE_2526_Metadata.xlsx': 'https://files.digital.nhs.uk/D2/087EB2/AE_2526_Metadata.xlsx',
    'hosp-epis-stat-outp-rep-tabs-2025-26-tab.xlsx': 'https://files.digital.nhs.uk/3A/DC44DE/hosp-epis-stat-outp-rep-tabs-2025-26-tab.xlsx',
    'hosp-epis-stat-outp-all-firs-atte-2025-26-data.csv': 'https://files.digital.nhs.uk/BC/881240/hosp-epis-stat-outp-all-firs-atte-2025-26-data.csv',
    'hosp-epis-stat-outp-eth-imd-dec-2025-26-data.csv': 'https://files.digital.nhs.uk/41/3F7E90/hosp-epis-stat-outp-eth-imd-dec-2025-26-data.csv',
    'hosp-epis-stat-outp-meta-2025-26-tab.xlsx': 'https://files.digital.nhs.uk/15/578BB6/hosp-epis-stat-outp-meta-2025-26-tab.xlsx',
}
for line in (c.BRONZE / 'msitae_monthly' / 'urls.txt').read_text().split():
    urls[line.rsplit('/', 1)[-1]] = line

rows = []
for p in sorted(c.BRONZE.rglob('*')):
    if p.is_file() and p.suffix in {'.csv', '.xls', '.xlsx'}:
        rows.append({'file': str(p.relative_to(c.BRONZE)), 'bytes': p.stat().st_size, 'sha256': c.sha256(p),
                     'source_url': urls.get(p.name),
                     'file_mtime_utc': datetime.fromtimestamp(p.stat().st_mtime, timezone.utc).isoformat(timespec='seconds')})
manifest = pd.DataFrame(rows)
manifest

,file,bytes,sha256,source_url,file_mtime_utc
0,ae/AE2526_CQI_Open_Data_AR.csv,4303505,fd14ae027dd6392bb947e741b6d06d3be4edf012c92cd7...,https://files.digital.nhs.uk/96/785BF6/AE2526_...,2026-10-03T09:50:23+00:00
1,ae/AE2526_ECDS_MSitAE_Tables.xlsx,195892,79a5d6158fc25492e70c2ad2744226a31a7145327f3933...,https://files.digital.nhs.uk/08/D5507A/AE2526_...,2026-10-03T09:50:24+00:00
2,ae/AE_2526_Metadata.xlsx,89400,192f5522f2b7a0a98c5895a1790cb1a52b139e1e018dd3...,https://files.digital.nhs.uk/D2/087EB2/AE_2526...,2026-10-03T09:50:24+00:00
3,msitae_monthly/April-2025-AE-by-provider-sfo2q...,436736,58d7a5a454c492c9e51a3adc161cbfd0265bfea20ad2bb...,https://www.england.nhs.uk/statistics/wp-conte...,2026-10-03T09:55:44+00:00
4,msitae_monthly/April-2025-CSV-revised.csv,28844,3f1985ad48750615b16eea59dd44512f154eba6e663e9b...,https://www.england.nhs.uk/statistics/wp-conte...,2026-10-03T09:55:45+00:00
5,msitae_monthly/August-2025-AE-by-provider-0G7W...,442880,4ba606f5354d7b17aad4ad219b3d93d6a8a3e317706deb...,https://www.england.nhs.uk/statistics/wp-conte...,2026-10-03T09:55:45+00:00
6,msitae_monthly/August-2025-CSV-revised.csv,29338,423930b9f916fc2e279a88804b37f17a0a9f9c37cb4953...,https://www.england.nhs.uk/statistics/wp-conte...,2026-10-03T09:55:45+00:00
7,msitae_monthly/December-2025-AE-by-provider-re...,437248,64ce1e8516e602797ea7361271e4842f6460baa6e1c24e...,https://www.england.nhs.uk/statistics/wp-conte...,2026-10-03T09:55:49+00:00
8,msitae_monthly/February-2026-AE-by-provider-lo...,437760,719ef01cfdacc7609b420001c15e2676e037f5f82dc7c6...,https://www.england.nhs.uk/statistics/wp-conte...,2026-10-03T09:55:48+00:00
9,msitae_monthly/February-2026-CSV-Dl8t54.csv,29315,87f870535ea397d60645e9551672ece8d5193e2d6bc8e0...,https://www.england.nhs.uk/statistics/wp-conte...,2026-10-03T09:55:48+00:00


In [3]:
# I want to know about any file I cannot trace back to a URL, so I check rather than assume.
untraced = manifest[manifest.source_url.isna()]
print(f'{len(manifest)} files; {len(untraced)} without a recorded source URL')
manifest.to_parquet(c.BRONZE / 'manifest.parquet', index=False)

31 files; 0 without a recorded source URL
